# Football Player Market Value Prediction

This notebook contains the full pipeline for predicting the market value (`market_value_in_eur`) of football players based on transfer records.

## 0. Environment Setup

Creates a `.venv` virtual environment using [uv](https://docs.astral.sh/uv/getting-started/installation/) and installs all required dependencies.

> **uv not installed?** Run the following in your terminal first:
> ```bash
> # Linux / macOS
> curl -LsSf https://astral.sh/uv/install.sh | sh
>
> # Windows (PowerShell)
> powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
> ```
> Full installation guide: https://docs.astral.sh/uv/getting-started/installation/

> **Before running the cell below**, bootstrap `ipykernel` into the venv from your terminal — this is required once so that VS Code can use the `.venv` kernel at all:
> ```bash
> uv venv .venv && uv pip install --python .venv/bin/python ipykernel
> ```
> Then select the `.venv` kernel in VS Code and run the cell to install the remaining packages.

In [3]:
import subprocess, sys

# Verify uv is available
result = subprocess.run(['uv', '--version'], capture_output=True, text=True)
if result.returncode != 0:
    raise EnvironmentError(
        'uv is not installed or not on PATH.\n'
        'Install it from: https://docs.astral.sh/uv/getting-started/installation/'
    )
print(f'uv found: {result.stdout.strip()}')

# Create .venv
subprocess.run(['uv', 'venv', '.venv', '--clear'], check=True)

# Install dependencies into the venv
packages = [
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'ipykernel',
]
subprocess.run(
    ['uv', 'pip', 'install', '--python', '.venv/bin/python', *packages],
    check=True,
)
print('\nAll dependencies installed successfully.')

uv found: uv 0.11.1 (x86_64-unknown-linux-gnu)

All dependencies installed successfully.


Using CPython 3.12.3 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 43 packages in 26ms
Installed 43 packages in 87ms
 + asttokens==3.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + executing==2.2.1
 + fonttools==4.62.1
 + ipykernel==7.2.0
 + ipython==9.12.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.19.2
 + joblib==1.5.3
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + kiwisolver==1.5.0
 + matplotlib==3.10.8
 + matplotlib-inline==0.2.1
 + nest-asyncio==1.6.0
 + numpy==2.4.4
 + packaging==26.0
 + pandas==3.0.2
 + parso==0.8.6
 + pexpect==4.9.0
 + pillow==12.2.0
 + platformdirs==4.9.6
 + prompt-toolkit==3.0.52
 + psutil==7.2.2
 + ptyprocess==0.7.0
 + pure-eval==0.2.3
 + pygments==2.20.0
 + pyparsing==3.3.2
 + python-dateutil==2.9.0.post0
 + pyzmq==27.1.0
 + scikit-learn==1.8.0
 + scipy==1.17.1
 + seaborn==0.13.2
 + six==1.17.0
 + stack-data==0.6.3
 + threadpo

## 1. Imports

Load all required libraries for data manipulation, visualization, and machine learning.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
pd.set_option('display.max_columns', None)

## 2. Load Dataset

Read the `transfers.csv` file containing 157,186 football transfer records. The target variable is `market_value_in_eur`.

In [5]:
df = pd.read_csv('transfers.csv')

print(f'Shape: {df.shape}')
df.head()

Shape: (157186, 10)


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,784335,2027-07-18,27/28,6505,6502,Gimcheon Sangmu,Jeonbuk Hyundai,0.0,500000.0,Jun-soo Byeon
2,402135,2027-07-04,27/28,6505,515,Gimcheon Sangmu,Without Club,NaN,350000.0,Jun-su Ahn
3,716435,2027-07-04,27/28,6505,30925,Gimcheon Sangmu,Gwangju FC,0.0,325000.0,Kang-hyun Lee
4,803933,2027-06-30,26/27,4467,14,TSV Hartberg,Austria Vienna,0.0,300000.0,Luca Pazourek
